## Annahmen (Stand 28.10.)
Alle frei gewählten Werte stehen in der Zelle darunter: Split, Schwelle, Recovery, Marge. Wer eine Annahme ändern will, ändert sie nur dort und führt das Notebook neu aus.

In [ ]:
# Annahmen — alle Stellschrauben an einer Stelle (zugleich Abschnitt „Annahmen" der Model Card)
# Ändern nur hier, danach alle Zellen neu ausführen (Run All).

# Datenaufteilung
test_anteil = 0.3               # Anteil Testdaten beim train_test_split (30/70)
seed = 42                       # random_state: macht die Zufallsauswahl reproduzierbar

# Entscheidung
decision_bound = 0.5            # Schwelle: ab dieser Ausfallwahrscheinlichkeit wird der Kredit abgelehnt

# Euro-Bewertung
recovery = 0.15                 # Recovery-Quote: Anteil, den die Bank nach einem Ausfall zurückholt
verlust_je_euro = 1 - recovery  # Verlust je ausgefallenem Euro (0,85); Ausfall am ersten Tag, keine Tilgung
marge_annual = 0.05             # Marge p. a. vor Risiko-, Gemein- und Kapitalkosten

In [ ]:
# Daten Import der Banking data - credit scoring project (date of data origination: Germany 1999)
# Ergänzung der Spalte "ausfall" basierend auf class von good/bad zu 0/1
from sklearn.datasets import fetch_openml

daten = fetch_openml("credit-g", version=1, as_frame=True)
df = daten.frame          # .frame = die ganze Tabelle inkl. Zielspalte "class"
df['ausfall'] = df['class'].map({'good': 0, 'bad': 1}).astype(int)
print(df.shape)
df.head(5)

In [ ]:
# Bereinigung der Input-Daten um die Ergebnis-Spalten 'class' und 'ausfall', um Datenleck zu vermeiden,
# i.e. Ergebnisse können aus den Inputdaten abgelesen werden, was das Modell direkt lernt 
# und somit perfekt die Losses erkennt aber in Wirklichkeit kein 'echtes' Muster gelernt hat.
# train_test_split: test_size=0.3 bedeutet Test- zu Trainingsdaten 30/70;random_state=42 macht die Zufallsauswahl reproduzierbar (42 frei gewählt)
# stratify=y erhält die Struktur zwischen Anteil Ausfälle, i.e. in Test und Trainingsdaten ca. 30% Ausfälle
import pandas as pd
from sklearn.model_selection import train_test_split

y = df['ausfall']
X = df.drop(columns=['class', 'ausfall'])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_anteil, random_state=seed, stratify=y)

print(X_train.shape, X_test.shape, y_test.shape, y_train.shape)
print(y_train.mean(), y_test.mean(), y_test.sum())

Zerlegung - Hot-One-Coding:
Die Kategorien in den Category-Spalten erstezen durch 0/1 für jede Ausprägung der jeweiligen Kategorien, z.B. Bei einer Kategorie mit 4 Ausprägungen, entstehen vier neue Spalten anstelle der einen alten mit der Information, welche Kategorie gilt für die jeweilige Zeile. Gleiches Vorgehen für X_Train und X_Test. Danach Prüfung, ob die Anzahl der Spalten und der Namen zwischen den beiden Datensätzen passen.

In [ ]:
X_train_k = pd.get_dummies(X_train, dtype=int)
X_test_k = pd.get_dummies(X_test, dtype=int)
print(X_train_k.shape)
print(X_test_k.shape)
print(X_train_k.shape[1] == X_test_k.shape[1])
print(list(X_train_k.columns) == list(X_test_k.columns))
print(X_train_k.drop(columns=X_train.select_dtypes('number').columns).sum(axis=1).unique())

Zerlegung:
Über fit() die Mittelwerte und skalierung der 7 Zahlenspalten auf Basis der Trainingsdaten ausrechnen
Über transform() die standardisierten werte für die Trainingsdaten und separat für die Testdaten berechnen
Dann das np.array wieder in ein df überführen. Über num_cols wird gesteuert, welche Spalten standardisiert werden (beachte, dass num_cols sich auf X_train mit ursprünglichen Spalten bezieht, damit nach Nummern gewählt werden kann, vor Hot-One-Codierung, sonst würde auch die Categorie-Zahlen standardisiert werden!)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_s = X_train_k.copy()
X_test_s = X_test_k.copy()

num_cols = X_train.select_dtypes(include='number').columns

X_train_s[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_s[num_cols] = scaler.transform(X_test[num_cols])

print(X_train_s.shape, X_test_s.shape, num_cols.shape)
print(y_train.shape, y_test.shape)
print(X_train_s[num_cols].mean().round(3)) # ergibt 0, weil Test auf den Standardisierten Daten
print(X_train_s[num_cols].std().round(2)) # ergibt 1, weil Test auf den Standardisierten Daten
print(X_test_s[num_cols].mean().round(3)) # Abweichung der Mittelwerte der Testdaten zu den Trainingsdaten
#print(X_train_k['duration'].head(3))
print(scaler.mean_)
#print(X_train_s['housing_own'].head(3))

## Zerlegung Teil 2c — eigene LogistischeRegression auf den Kreditdaten

Ziel: Meine Klasse vom 09.10. lernt aus 700 Krediten mit 61 Merkmalen,
ob ein Kredit ausfällt. Am Ende: Gewichte, Kostenverlauf, Accuracy auf dem Testteil.

1. Tabellen in Zahlen-Arrays verwandeln — meine Klasse rechnet mit Matrizen,
   nicht mit DataFrames. Ergebnis: Xtr (700, 61), ytr (700,), Xte (300, 61), yte (300,).
2. Klasse unverändert übernehmen — sie liest die Spaltenzahl selbst aus X.shape,
   deshalb gilt sie für 2 Merkmale wie für 61.
3. Objekt anlegen mit Lernrate 0,1 und 1000 Schritten, dann fit auf Xtr/ytr.
   Ergebnis: w (61,) — ein Gewicht je Spalte; b eine Zahl; J_verlauf Liste mit 1000 Kosten.
4. Prüfen, ob gelernt wurde: Kosten am Anfang (0,693 = analog zu MÜnzwurf, da auch p = 0.5) gegen Kosten am Ende,
   und ob kein Schritt die Kosten erhöht hat (größte Differenz ≤ 0).
5. Vorhersage auf Xte, Vergleich mit yte, Anteil der Treffer = Accuracy.
   Maßstab: 0,70 wäre „immer good" — alles darüber hat das Modell gelernt.

In [ ]:
# Eigene Logistische Regression gebaut als Python "class"
import numpy as np

# Mein eigener Code erwartet heir Numpy arrays, weil ich die Codierung aus einer früheren Übung basierend auf Numpy übernommen habe.
# Daher die Umwandlung vom DataFrame oben zu den np.arrays hier im Code. (siehe Kommentare in der Zerlegung)
Xtr = X_train_s.values
ytr = y_train.values
Xte = X_test_s.values
yte = y_test.values

class LogistischeRegression:
    def __init__(self, alpha=0.1, n_iter=1000):
        # Hyperparameter speichern; self.w = None, self.b = None, self.J_verlauf = []
        self.alpha = alpha
        self.n_iter = n_iter
        self.w = None
        self.b = None
        self.J_verlauf = []

    def sigmoid(self, z):
        # 1 / (1 + np.exp(-z))
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        m, n = X.shape
        w = np.zeros(n) 
        b = 0.0
        for i in range(self.n_iter):
            p = self.sigmoid(X @ w + b)
            err = p - y
            w -= self.alpha * (X.T @ err) / m #Gradient Descent w
            b -= self.alpha * err.mean() #Gradient Descent b
            J = -np.mean(y*np.log(p) + (1-y)*np.log(1-p))
            self.J_verlauf.append(J)
        self.w = w
        self.b = b

        return self

    def predict_proba(self, X):
        return self.sigmoid(X @ self.w + self.b)

    def predict(self, X):
        return (self.predict_proba(X) >= decision_bound).astype(int)


eigen = LogistischeRegression(alpha=0.1, n_iter=1000).fit(Xtr, ytr)
eigen2 = LogistischeRegression(alpha=0.5, n_iter=1000).fit(Xtr, ytr)
d = np.diff(eigen.J_verlauf)
#print(eigen.w.shape, eigen.w, round(eigen.b, 3), round(eigen.J_verlauf[0], 3), round(eigen.J_verlauf[-1], 3))
#print(round(d.max(), 6), np.all(d <= 0))

print(round(eigen2.J_verlauf[9], 3), round(eigen2.J_verlauf[99], 3),round(eigen2.J_verlauf[-1], 3))
print(round(eigen.J_verlauf[9], 3), round(eigen.J_verlauf[99], 3),round(eigen.J_verlauf[-1], 3))
print((eigen.predict(Xte) == yte).mean())
print((eigen.predict(Xtr) == ytr).mean())
print(eigen.predict_proba(Xtr).shape, eigen.predict_proba(Xte).shape)


## Zerlegung Teil 2d — sklearn LogisticRegression auf denselben Daten

Ziel: Dasselbe Modell aus der Bibliothek, dann Vergleich mit meiner Klasse —
gleiche Vorzeichen? gleiche Accuracy?

1. Klasse LogisticRegression aus sklearn.linear_model laden (import) und ein Objekt sk
   anlegen mit Vorgabe „höchstens 1000 Schritte". Ergebnis: leeres Modell, noch ohne coef_.
2. fit auf X_train_s (700, 61) und y_train (700,) — DataFrames gehen direkt, kein .values.
   fit speichert im Objekt: coef_ (1, 61), intercept_ eine Zahl, n_iter_ die gebrauchten Schritte.
   Rückgabe ist das Objekt selbst.
3. score auf X_test_s (300, 61) und y_test (300,) → eine Zahl, die Accuracy.
   Daneben stellen: eigen 0,777 und „immer good" 0,70.
4. Vergleichstabelle: Zeilen = 61 Spaltennamen aus X_train_s.columns,
   Spalte eigen = eigen.w (61,), Spalte sklearn = coef_[0] (61,) — die [0] holt die Zeile
   aus der (1, 61)-Form. Ergebnis: DataFrame (61, 2).
5. Vorzeichen beider Spalten vergleichen, Treffer zählen → eine Zahl von 61.
   Erwartung: fast alle gleich, Beträge verschieden (sklearn regularisiert, C=1.0).

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression

sk = LogisticRegression(max_iter=1000)
sk.fit(X_train_s, y_train)

print("Koeffizienten_Form:", sk.coef_.shape)
print("Intercept:", sk.intercept_)
print("Klassen:", sk.classes_)
print("Iterationen:", sk.n_iter_)
print("Accuracy Train:", sk.score(X_train_s, y_train))
print("Accuracy Test:", sk.score(X_test_s, y_test))

vergleich = pd.DataFrame({'eigen': eigen.w, 'sklearn': sk.coef_[0]}, index=X_train_s.columns)
same_sign = (np.sign(vergleich['eigen']) == np.sign(vergleich['sklearn'])).sum()
rel_sign = (np.sign(vergleich['eigen']) == np.sign(vergleich['sklearn'])).mean()
print(vergleich)
print(same_sign, rel_sign)

abweichler = vergleich[np.sign(vergleich['eigen']) != np.sign(vergleich['sklearn'])]
print(abweichler.round(3))

Zerlegung zu Confusion Matrix:
1. Vier Zahlen der Vierfeldertafel:  
   Feld1: Tatsächlich gut & Modell gut
   => y_test == 0 & predict == 0 mit ((y == 0) & (pred == 0)).sum()
   Feld2: Tatsächlich Ausfall & Modell Ausfall
   => y_test == 1 & predict == 1 mit ((y == 1) & (pred == 1)).sum()
   Feld3: Modell gut & Tatsächlich Ausfall
   => predict == 0 & y_test == 1 mit ((y == 1) & (pred == 0)).sum()
   Feld4: Modell Ausfall & Tatsächlich gut
   => predict == 1 und y_test == 0 mit ((y == 0) & (pred == 1)).sum()
2. Anzahl richtig vorhergesagter Ausfälle / Anzahl gesamte tatsächliche. 
   Ausfälle gesamt (Recall); Anzahl richtig vorhergesagter Ausfällt / Anzahl vorhergesagter Ausfälle gesamt (Precision)
3. Genauigkeit/Accuracy über beide Wege

In [ ]:
# Confustion Matrix basierend auf eigen:
pred_eigen = eigen.predict(Xte)
conf_1_eig = ((yte == 0) & (pred_eigen == 0)).sum()
conf_2_eig = ((yte == 0) & (pred_eigen == 1)).sum()
conf_3_eig = ((yte == 1) & (pred_eigen == 0)).sum()
conf_4_eig = ((yte == 1) & (pred_eigen == 1)).sum()
print(f'eig-model: Real-good Pred-good: {conf_1_eig}, Real-good Pred-bad: {conf_2_eig}, Real-bad Pred-good: {conf_3_eig}, Real-bad Pred-bad : {conf_4_eig}')

# Confustion Matrix basierend auf sklearn:
#pred_sk = sk.predict(X_test_s) geändert in die beiden Zeilen darunten, um die Desicion Boundary flexibel setzen zu können, e.g. 0.3 anstatt 0.5
p_ausfall = sk.predict_proba(X_test_s)[:, 1]
pred_sk = (p_ausfall >= decision_bound).astype(int)
conf_1_sk = ((y_test == 0) & (pred_sk == 0)).sum()
conf_2_sk = ((y_test == 0) & (pred_sk == 1)).sum()
conf_3_sk = ((y_test == 1) & (pred_sk == 0)).sum()
conf_4_sk = ((y_test == 1) & (pred_sk == 1)).sum()
print(f'conf_sk-model: Real-good Pred-good: {conf_1_sk}, Real-good Pred-bad : {conf_2_sk}, Real-bad Pred-good: {conf_3_sk}, Real-bad Pred-bad: {conf_4_sk}')


recall_eigen = ((pred_eigen == 1) & (yte == 1)).sum() / (yte == 1).sum()
precision_eigen = ((pred_eigen == 1) & (yte == 1)).sum() / (pred_eigen == 1).sum()

recall_sk = ((pred_sk == 1) & (y_test == 1)).sum() / (y_test == 1).sum()
precision_sk = ((pred_sk == 1) & (y_test == 1)).sum() / (pred_sk == 1).sum()

print(f'Recall-eigen: {recall_eigen:.1%}, Recall_sk: {recall_sk:.1%}')
print(f'Precision-eigen: {precision_eigen:.1%}, Precision_sk: {precision_sk:.1%}')

from sklearn.metrics import confusion_matrix
from sklearn.metrics import recall_score, precision_score

print(f'Confusion Matrix: {confusion_matrix(y_test, pred_sk)}')
print(f'Recall-Score: {recall_score(y_test, pred_sk):.1%}, Precision-Score: {precision_score(y_test, pred_sk):.1%}')

maske = (pred_sk == 0) & (y_test == 1)
ne_ausfall = X_test.loc[maske, 'credit_amount'].sum() * verlust_je_euro
print(f"Nicht erkannte Ausfälle des Modells: {ne_ausfall:,.0f}")
print(f"Gesamt-Kreditsumme: {X_test['credit_amount'].sum():,.0f}")

maske2 = (y_test == 1)
gesamt_ausfall = X_test.loc[maske2, 'credit_amount'].sum() * verlust_je_euro
print(f"Gesamtausfall: {gesamt_ausfall:,.0f} EUR")
print(f"Gesamtausfall ex Recoveries: {gesamt_ausfall / verlust_je_euro:,.1f} EUR")


model_adv_abs = gesamt_ausfall - ne_ausfall
print(f'Der Vorteil mit Modell vs. ohne Modell (abs): {model_adv_abs:,.0f}')

model_adv_rel = model_adv_abs / gesamt_ausfall
print(f'Der Vorteil mit Modell vs. ohne Modell (rel): {model_adv_rel:.1%}')

maske3 = (pred_sk == 1) & (y_test == 0)
lost_volume = X_test.loc[maske3, ['credit_amount', 'duration']]
lost_revenues = (lost_volume['credit_amount'] * lost_volume['duration'] * marge_annual / 12).sum()
print(f"Entgangenes Kreditvolumen: {lost_volume['credit_amount'].sum():,.1f} EUR")
print(f"Durchschnittliche Laufzeit: {lost_volume['duration'].mean():,.1f} Monate")

gewichtete_laufzeit = np.average(lost_volume['duration'],weights=lost_volume['credit_amount'])
print(f"Volumengewichtete Laufzeit: {gewichtete_laufzeit:.1f} Monate")

print(f"Entgangene Erträge aufgrund inkrementeller Ablehnung: {lost_revenues:,.1f} EUR")

performing_volume = X_test.loc[(y_test == 0), ['credit_amount', 'duration']]
revenues = (performing_volume['credit_amount'] * performing_volume['duration'] * marge_annual / 12).sum()
print(f"Performing volume: {performing_volume['credit_amount'].sum():,.1f} EUR")
print(f"Gesamterträge der performing loans: {revenues:,.1f} EUR")


In [ ]:
# Schritt 1 — Portfolio-Ergebnis: was das genehmigte Portfolio tatsächlich verdient (ergänzt 28.10.)
# Mein Netto-Beitrag oben misst den Abstand zu "alle genehmigen" (Basis: -275.700 EUR).
# Hier das absolute Ergebnis der Kredite, die das Modell bei decision_bound durchlässt.
ertrag_genehmigt = revenues - lost_revenues     # Erträge der guten Kredite, die genehmigt werden
verlust_genehmigt = ne_ausfall                  # Verluste der Ausfälle, die das Modell durchlässt
portfolio = ertrag_genehmigt - verlust_genehmigt
print(f"Schwelle: {decision_bound}")
print(f"Ertrag genehmigt: {ertrag_genehmigt:,.0f} EUR")
print(f"Verlust genehmigt: {verlust_genehmigt:,.0f} EUR")
print(f"Portfolio-Ergebnis: {portfolio:,.0f} EUR")

# Break-even-Ausfallquote: bis zu welcher Ausfallwahrscheinlichkeit trägt sich ein Kredit?
# Ertrag je Euro Kredit (gute Kredite) gegen Verlust je Euro Kredit (verlust_je_euro aus der Annahmen-Zelle)
ertrag_je_euro = revenues / performing_volume['credit_amount'].sum()
pd_break_even = ertrag_je_euro / (ertrag_je_euro + verlust_je_euro)
print(f"Ertrag je Euro Kredit: {ertrag_je_euro:.1%}")
print(f"Break-even-Ausfallquote: {pd_break_even:.1%} (Datensatz: {y_test.mean():.0%})")

In [ ]:
top3_namen = vergleich['sklearn'].abs().sort_values(ascending=False).head(3).index
top3 = vergleich['sklearn'].loc[top3_namen]
print(f'Die Top-3-Gewichte sind folgende:\n{top3.round(3)}')

In [ ]:
# Detaillierter Weg (mein initialer Ansatz)
# zweck_anzahl = df['purpose'].value_counts()
# ausfall_pro_zweck = df.loc[df['ausfall'] == 1, 'purpose'].value_counts()
# ausfall_quote_zweck = ausfall_pro_zweck / zweck_anzahl
# ausfall_quote_gesamt = ausfall_pro_zweck / (df['ausfall'] == 1).sum()
# print(f'Anzahl der verschiedenen Kreditzwecke:\n{zweck_anzahl}')
# print(ausfall_pro_zweck)
# print(f'Anteil Ausfälle innerhalb eines Kreditzweckes:\n{ausfall_quote_zweck.round(2)}')
# print(f'Anteil Ausfälle eines Kreditzwecks an den Gesamtausfällen:\n{ausfall_quote_gesamt.round(3)}')

# Alternative, sehr effizient:
df.groupby('purpose')['ausfall'].agg(anzahl='count', quote='mean').sort_values('quote', ascending=False)

,anzahl,quote
purpose,,
education,50,0.440000
other,12,0.416667
new car,234,0.380342
repairs,22,0.363636
business,97,0.350515
domestic appliance,12,0.333333
furniture/equipment,181,0.320442
radio/tv,280,0.221429
used car,103,0.165049


### Deutung der drei größten Gewichte (02.11., Tag 68)

Betragsgrößte Gewichte des sklearn-Modells (`vergleich['sklearn']`): alle drei sind Null-Eins-Schalter aus `get_dummies`, keine Zahlenspalte.

| Merkmal | Gewicht | Richtung | Hausverstand | Vorbehalt |
|---|---|---|---|---|
| `purpose_education` (Kreditzweck Ausbildung) | +1,00 | erhöht Ausfallwahrscheinlichkeit | plausibel: nach der Ausbildung kein Arbeitsplatz, geringes Einkommen während der Laufzeit — deckt sich mit Praxiserfahrung; strukturell gut aufgesetzt kann es trotzdem ein attraktives Geschäftsfeld sein | **nur 50 von 1000 Krediten**, Ausfallquote 44 % (Gesamt 30 %). Fünf Ausfälle mehr oder weniger → 34 % oder 54 %. Trägt keine Ablehnung allein, höchstens einen manuellen Prüfhinweis |
| `checking_status_no checking` (kein Girokonto bei der Bank bekannt) | −0,92 | senkt Ausfallwahrscheinlichkeit | intuitiv falsch herum | **Selektionseffekt**: Datensatz enthält nur *vergebene* Kredite. Wer ohne Konto genehmigt wurde, stand sonst sehr gut da (oder führt sein Hauptkonto woanders). Gelernt wird „kein Konto *und trotzdem genehmigt* = sicher" — taugt nicht, um neue Anträge ohne Konto großzügiger zu behandeln |
| `credit_history_critical/other existing credit` (kritische Historie / andere bestehende Kredite) | −0,80 | senkt Ausfallwahrscheinlichkeit | widerspricht der Risiko-Sicht | gleicher Selektionseffekt, schärfer: wer mit kritischer Historie noch einen Kredit bekam, wurde besonders streng geprüft; die Überlebenden fallen seltener aus. **Das Modell beschreibt die Ausfälle unter der alten Kreditpolitik, nicht das Risiko des Merkmals an sich** |

**Warum keine Zahlenspalte in den Top 3:** Laufzeit (+0,36) heißt „eine Standardabweichung mehr Laufzeit (≈ 12 Monate) → Score +0,36". Ausbildung (+1,0) heißt „Schalter an statt aus". Ein Schalter springt immer um den vollen Betrag, eine Zahlenspalte meist nur um Bruchteile einer Standardabweichung — deshalb stehen Schalter mit kleinen Gruppen oben, obwohl Laufzeit und Rate über *alle* Kunden hinweg mehr erklären. Regel: Zahlengewichte untereinander vergleichen, Schaltergewichte untereinander, nicht gemischt.

**Prüfbau zu Ausbildung:** `df['purpose'].value_counts()` und Ausfallquote je Zweck (Zelle oben) → education 50 Fälle, Quote 0,44.

→ Rohstoff für den Abschnitt „Grenzen" der Model Card (Di 10.11.): Selektionseffekt als eigene Einschränkung aufnehmen; kleine Gruppen mit großen Gewichten kennzeichnen.


### Herleitung Break-even-Ausfallquote (ergänzt 02.11., Tag 68)

**Kernaussage:** Die Break-even-Quote ist die Ausfallwahrscheinlichkeit p, bei der ein Kredit *im Erwartungswert* genau null bringt — erwarteter Ertrag und erwarteter Verlust halten sich die Waage.

**An den Zahlen, pro verliehenem Euro:**
- Ein guter Kredit bringt `ertrag_je_euro` = 66 398 / 628 148 ≈ **10,6 Cent** (Marge über die Laufzeit).
- Ein Ausfall kostet `verlust_je_euro` = 1 − Recovery = **85 Cent**.

Für einen Kunden mit Ausfallwahrscheinlichkeit p:
- Erwarteter Ertrag = (1 − p) × 10,6 Cent — nur wenn er *nicht* ausfällt, zahlt er die Marge.
- Erwarteter Verlust = p × 85 Cent — nur wenn er ausfällt, verliert die Bank.

| p | Erwarteter Ertrag | Erwarteter Verlust | Saldo je Euro |
|---|---|---|---|
| 30 % (Datensatz) | 0,70 × 10,6 = 7,4 ct | 0,30 × 85 = 25,5 ct | **−18,1 ct** → kein Portfolio profitabel |
| 11,1 % (Break-even) | 0,889 × 10,6 = 9,4 ct | 0,111 × 85 = 9,4 ct | **0** |
| 5 % | 0,95 × 10,6 = 10,1 ct | 0,05 × 85 = 4,3 ct | **+5,8 ct** |

**Formel:** Ertrag und Verlust gleichsetzen und nach p auflösen:

```
(1 − p) · e = p · l
e − p·e     = p·l
e           = p · (e + l)
p           = e / (e + l)
```

→ `pd_break_even = ertrag_je_euro / (ertrag_je_euro + verlust_je_euro)` = 0,106 / (0,106 + 0,85) = **11,1 %**.

**Lesart:** Der Nenner (e + l) ist, was zwischen bestem und schlechtestem Fall auf dem Spiel steht — Marge plus Verlust. Der Zähler e ist der Gewinnanteil daran. Je kleiner die Marge im Verhältnis zum Verlust, desto näher rückt die tragbare Ausfallquote an null: mit 5 % Marge p. a. und 85 % Verlust bleibt nur rund ein Zehntel Spielraum.

**Vereinfachung (→ Model Card, Einschränkung 5):** Die Formel nimmt an, dass ein Ausfallkunde *gar keine* Marge zahlt (Ausfall am ersten Tag). Real zahlt er bis zum Ausfall Zinsen; der echte Break-even läge etwas höher.


### Deutung Ertragsseite (Stand 28.10.)

**Ergebnis je Schwelle** (Tsd. EUR, sklearn-Modell, Testteil 300 Kredite; „Portfolio-Ergebnis" = Ertrag der genehmigten guten Kredite − Verlust der genehmigten Ausfälle):

| Schwelle | Modellnutzen | Entgangene Erträge | Netto-Beitrag (vs. alle genehmigen) | Portfolio-Ergebnis |
|---|---|---|---|---|
| 0,8 | 43,8 | −2,7 | 41,1 | −234,6 |
| 0,7 | 66,8 | −5,9 | 60,9 | −214,8 |
| 0,5 | 203,2 | −10,3 | 192,9 | −82,8 |
| 0,3 | 246,7 | −22,3 | 224,4 | −51,3 |
| 0,1 | 315,0 | −53,4 | 261,6 | −14,1 |
| 0,01 | 342,1 | −65,9 | 276,2 | ≈ 0 (fast niemand genehmigt) |

Basis „alle genehmigen": Ertrag 66,4 − Verlust 342,1 = **−275,7**. Der Netto-Beitrag ist das Portfolio-Ergebnis + 275,7 — gleiche Reihenfolge, aber jede Zeile sieht nach Gewinn aus, obwohl keine Schwelle Geld verdient.

**Warum keine Schwelle Geld verdient — Break-even-Ausfallquote:** Ein guter Kredit bringt über die Laufzeit ca. 10,6 % seines Betrags (66,4 / 628,1), ein Ausfall kostet 85 %. Ein Kredit trägt sich also nur bis zu einer Ausfallwahrscheinlichkeit von ca. **11 %**. Der Datensatz hat **30 %** — im Schnitt vernichtet jeder Kredit Wert, und das Modell findet kaum eine Gruppe, die sicher genug ist.

**Der eine Satz:** Das Modell trennt brauchbar (Recall 53 % bei 0,5, 62 % bei 0,4), aber bei 30 % Ausfällen und 5 % Marge ist kein Portfolio profitabel — die Entscheidung „wo die Schwelle liegt" braucht realistische Ausfallquoten und Preise.

**Konsistenz-Hinweis:** Ertrag und Verlust stehen beide auf vollem Kreditbetrag (Ertrag über die volle vertragliche Laufzeit, Verlust als Ausfall am ersten Tag). Werden beide Seiten auf die halbe mittlere Laufzeit gesetzt (Faktor 0,5), bleibt die Break-even-Quote bei ca. 11 % — wichtig ist, beide Seiten gleich zu behandeln.

### Einschränkungen (für die Model Card, 13.11.)

1. **Stichprobe nicht repräsentativ:** German Credit (Deutschland, 1990er) hat 30 % Ausfälle; Ausfälle sind in diesem Datensatz überrepräsentiert. Echte Konsumentenkredit-Portfolios liegen typischerweise im niedrigen einstelligen Prozentbereich.
2. **Wahrscheinlichkeiten nicht kalibriert:** `predict_proba` ist auf eine Ausfallquote von 30 % geeicht und lässt sich nicht direkt als Ausfallwahrscheinlichkeit eines echten Portfolios lesen. Umrechnung (Kalibrierung) → Fenster 2 (Register A6).
3. **Marge und Ausfallquote passen nicht zusammen:** 5 % Marge vor Risikokosten ist ein Preis für ein Portfolio mit niedriger Ausfallquote; bei 30 % würde eine Bank den Risikoaufschlag einpreisen oder nicht verleihen. Die Euro-Rechnung zeigt deshalb die Mechanik, nicht eine realistische Wirtschaftlichkeit.
4. **Kleiner Testteil:** 300 Kredite, 90 Ausfälle — ein Ausfall verschiebt den Recall um 1,1 Prozentpunkte; Kennzahlen schwanken mit dem `random_state` des Splits. Stabile Bewertung über Kreuzvalidierung → Credit-Scoring v2.
5. **Vereinfachte Euro-Annahmen:** Verlust = Betrag × 0,85 (Ausfall am ersten Tag, keine Tilgung), Ertrag = Betrag × 5 % × Laufzeit (Marge vor Risiko-, Gemein- und Kapitalkosten).

### Ergebnisse mit 30% Testdaten und underlying desicion boundary of 0.5 als Referenz zum Vergleicfh mit Änderung von Decision Boundary (0.4) und Train_Test_Datensplit
 
eig-model: Real-good Pred-good: 185, Real-good Pred-bad: 25, Real-bad Pred-good: 42, Real-bad Pred-bad : 48. 
conf_sk-model: Real-good Pred-good: 187, Real-good Pred-bad : 23, Real-bad Pred-good: 42, Real-bad Pred-bad: 48. 
Recall-eigen: 53.3%, Recall_sk: 53.3%. 
Precision-eigen: 65.8%, Precision_sk: 67.6%. 
Confusion Matrix: [[187  23]
 [ 42  48]]. 
Recall-Score: 53.3%, Precision-Score: 67.6%. 
Nicht erkannte Ausfälle des Modells: 138,904. 
Gesamt-Kreditsumme: 1,030,647. 
Gesamtausfall: 342,124 EUR. 
Gesamtausfall ex Recoveries: 402,499.0 EUR. 
Der Vorteil mit Modell vs. ohne Modell (abs): 203,221. 
Der Vorteil mit Modell vs. ohne Modell (rel): 59.4%. 
Entgangenes Kreditvolumen: 85,440.0 EUR. 
Durchschnittliche Laufzeit: 23.6 Monate. 
Volumengewichtete Laufzeit: 28.8 Monate. 
Entgangene Erträge aufgrund inkrementeller Ablehnung: 10,269.4 EUR. 
Performing volume: 628,148.0 EUR. 
Gesamterträge der performing loans: 66,398.4 EUR

### Stand Tag 63 (Mo 19.10.) — Daten modellfertig

**Kette:** df (1000, 22) → Split → One-Hot-Kodierung → Standardisierung.
Reihenfolge zwingend: erst teilen, dann alles Weitere nur aus dem Trainingsteil lernen — sonst sieht das Modell Testdaten (Datenleck).

**Tabellen:**
- y = Spalte `ausfall` (1000,), 0/1. `class` und `ausfall` aus X entfernt, damit keine Ergebniswerte in den Input-Daten stehen.
- X_train (700, 20), X_test (300, 20), y_train, y_test — aus `train_test_split`, 70/30, `stratify=y` (homogene Datengrundlage: Ausfallquote 0,30 in beiden Teilen, 90 Test-Ausfälle), `random_state=42`.
- X_train_k (700, 61), X_test_k (300, 61) — aus `get_dummies`: 13 Kategorie-Spalten → 54 Null-Eins-Spalten, 7 Zahlenspalten unverändert.
- X_train_s (700, 61), X_test_s (300, 61) — aus `StandardScaler`: 7 Zahlenspalten mit Mittelwert/Streuung aus **Train** umgerechnet (fit nur auf Train, transform auf beide), 54 Null-Eins-Spalten unverändert.

**Modellfertig für morgen:** X_train_s, X_test_s, y_train, y_test.

**Der eine Satz:** `fit` lernt Zahlen und speichert sie mit Unterstrich (`mean_`, `coef_`); `transform` und `predict` benutzen diese Zahlen und geben etwas Neues zurück, ohne etwas zu speichern.

**Gestolpert heute:** (1) geglaubt, `fit` verändere die Daten bzw. speichere Vorhersagen — tut es nicht; (2) Standardisierung auf X_train statt X_train_k aufgesetzt, Kontrollwert 20 statt 61 nicht gegen den Sollwert geprüft.

### Stand Tag 64 (Di 20.10.) — zwei Modelle trainiert, Accuracy gegen Nichtstun

**Was steht:**
- `eigen` — meine Klasse `LogistischeRegression` (alpha 0,1, 1000 Schritte), trainiert auf
  `Xtr` (700, 61) / `ytr` (700,) — Arrays aus `X_train_s.values`, `y_train.values`.
  Gelernt: `eigen.w` (61,), `eigen.b` = −0,21 (eine Zahl), `eigen.J_verlauf` (Liste, 1000 Kosten).
- `sk` — sklearn `LogisticRegression(max_iter=1000)`, trainiert direkt auf den DataFrames
  `X_train_s` / `y_train`. Gelernt (Unterstrich = von fit gespeichert): `coef_` (1, 61),
  `intercept_` = −1,56, `n_iter_` = 53 Schritte bis Konvergenz.
- `vergleich` (61, 2) — beide Gewichtsvektoren nebeneinander, Zeilen = Spaltennamen.

**Zahlen, die man sich merkt:**
- Kosten Start 0,693 = ln 2 = Münzwurf (w = 0 → jede Wahrscheinlichkeit 0,5); Ende 0,454;
  kein Schritt bergauf (größte Differenz −8e-06).
- Accuracy Test: „immer good" 0,70 (210/300) · eigen 0,777 (233) · sklearn 0,783 (235).
  Modelle untereinander: kein Unterschied (2 Kredite). Gegen Nichtstun: ~23 Kredite mehr richtig.
  Aber: „immer good" erkennt 0 der 90 Ausfälle — das misst Accuracy nicht → Do Confusion Matrix.
- Vorzeichen: 55 von 61 gleich (0,90); die 6 Abweichler haben Beträge ≤ 0,09 — keine inhaltliche
  Meinungsverschiedenheit. Beträge verschieden, weil sklearn regularisiert (C = 1,0) und
  mit L-BFGS bis zur Konvergenz rechnet statt 1000 Gradientenschritte.
- Erste sieben Gewichte = die sieben Zahlenspalten: Laufzeit +0,35, Rate +0,34, bestehende
  Kredite +0,21, Betrag +0,19 → riskanter; Alter −0,09 → sicherer. Deutung am Do.

**Was gewackelt hat (Denkmodell-Fehler heute: 1):**
- Behälter vs. Inhalt: `np.sign('eigen')` → TypeError; `'eigen'` ist der Name, `vergleich['eigen']`
  die Spalte. Gleiches Muster wie `b.shape` (b ist eine Zahl, keine Form) — Tastversuch, kein Fehler.
- `coef_` ist (1, 61), zweidimensional mit einer Zeile; `pd.DataFrame` will (61,) → `coef_[0]` oder `.ravel()`.
  Grund: sklearn hält eine Zeile je Klasse bereit, bei zwei Klassen bleibt es eine.
- Name `df` war schon die Kredittabelle → Vergleichstabelle heißt `vergleich`. Ein Name pro Behälter.
- Zerlegung zweimal zu dünn (übersprungen / ohne Formen) — Benchmark: Eingabe → was passiert → Ergebnis mit Form.

**Offen bis Do 22.10.:** Confusion Matrix, Precision/Recall für die 90 Ausfälle · `predict_proba`,
Schwelle 0,5 vs. 0,3 · Deutung der drei größten Gewichte · Feynman-Test · Abschluss-Stand vor dem Urlaub.
